# 🔗 Task 2 – Deep Technical Blog on LangChain

**Objective:** Hands-on implementation of LangChain core components

**Components Covered:**
- Prompt Templates
- Chat Prompt Templates
- Memory
- Input Validation
- Prompt Generator
- Template Reusability

---

### What is LangChain?

**LangChain** is an open-source Python framework that helps developers build applications powered by Large Language Models (LLMs). Instead of writing complex boilerplate code to manage prompts, memory, and tool integrations — LangChain provides clean, reusable components that you can plug together like building blocks.

**Pipeline Flow:**
```
User Input → Prompt Template → LLM → Chain → Tool/Agent → Output
```

## Step 0: Install Required Libraries

We install LangChain along with its core and OpenAI integration packages.

In [ ]:
!pip install langchain==0.1.17 langchain-core langchain-openai

## Step 1: Import Libraries

We import all necessary LangChain components:

| Import | Purpose |
|---|---|
| `PromptTemplate` | Create reusable prompts with variables |
| `ChatPromptTemplate` | Create structured chat-style prompts |
| `OpenAI` | Connect to OpenAI LLM |
| `LLMChain` | Chain a prompt with an LLM |
| `ConversationBufferMemory` | Store conversation history |
| `initialize_agent` | Create an autonomous AI agent |
| `Tool` | Define tools that agents can use |

In [2]:
# Import all required LangChain components
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_openai import OpenAI
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent
from langchain.tools import Tool

---
## Example 1: Prompt Template

### What is a Prompt Template?

A **Prompt Template** is a reusable prompt with dynamic placeholders. Instead of writing the same prompt over and over with different values, you define it once and fill in the variables at runtime.

**Why it exists:**  
In real applications, prompts change based on user input. Templates keep your code clean, DRY (Don't Repeat Yourself), and easy to maintain.

**Example:**
```
Template : "Explain {topic} in simple terms"
Input    : topic = "LangChain"
Output   : "Explain LangChain in simple terms"
```

In [3]:
# Prompt Template Example
# PromptTemplate takes a list of variables and a template string
print("\nPrompt Template Example:")

template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)

# .format() fills in the variable and returns the final prompt string
print(template.format(topic="LangChain"))


Prompt Template Example:
Explain LangChain in simple terms


---
## Example 2: Memory

### What is Memory in LangChain?

By default, LLMs are **stateless** — they forget everything after each call. If you tell the LLM your name and then ask it in the next message, it won't remember.

**Memory** solves this by storing the conversation history and injecting it into every new prompt automatically.

### Types of Memory:
| Type | Description |
|---|---|
| `ConversationBufferMemory` | Stores the **full** conversation history |
| `ConversationBufferWindowMemory` | Stores only the **last N** messages |
| `ConversationSummaryMemory` | Stores a **summary** of old conversations (saves tokens) |

In this example, we use `ConversationBufferMemory` — the simplest type.

In [4]:
# Memory Example
# ConversationBufferMemory stores the full conversation as a string
print("\nMemory Example:")

memory = ConversationBufferMemory()

# save_context() manually stores a turn of conversation
# In a real app this happens automatically through ConversationChain
memory.save_context({"input": "Hello"}, {"output": "Hi there!"})
memory.save_context({"input": "What is AI?"}, {"output": "AI stands for Artificial Intelligence"})

# load_memory_variables() retrieves the stored history
print(memory.load_memory_variables({}))


Memory Example:
{'history': 'Human: Hello\nAI: Hi there!\nHuman: What is AI?\nAI: AI stands for Artificial Intelligence'}


**Output Explanation:**

The memory stores each conversation turn as:
```
Human: Hello
AI: Hi there!
Human: What is AI?
AI: AI stands for Artificial Intelligence
```
This history gets automatically injected into the next prompt so the model remembers what was said earlier.

---
## Example 3: Chat Prompt Template

### What is a Chat Prompt Template?

While `PromptTemplate` works with plain text, `ChatPromptTemplate` structures prompts as a **conversation** — with a system message and user messages.

This is important for **Chat Models** (like GPT-3.5-turbo) which expect a list of messages, not just a plain string.

**Structure:**
```
System  : Sets the AI's behavior/persona ("You are a helpful assistant")
User    : The actual question or instruction
```

**Why it matters:** The system message controls how the AI responds — you can make it a teacher, a code reviewer, a customer support agent, etc.

In [5]:
# Chat Prompt Template Example
# from_messages() takes a list of (role, content) tuples
print("\nChat Prompt Example:")

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),  # Sets the AI's persona
    ("user", "Explain {topic}")                 # Dynamic user message
])

# .format() fills in the {topic} variable
print(chat_template.format(topic="Neural Networks"))


Chat Prompt Example:
System: You are a helpful assistant
Human: Explain Neural Networks


---
## Example 4: Input Validation

### Why do we need Input Validation?

In real LangChain applications, prompts depend on user input. If a user provides an invalid value (like `"kids"` instead of `"beginner"`), the prompt might behave unexpectedly.

**Input validation** ensures that only accepted values reach the prompt template — making the application robust and predictable.

**Valid values in this example:**
- `audience` → `"beginner"`, `"intermediate"`, `"expert"`
- `tone` → `"formal"`, `"casual"`, `"fun"`

Any invalid input automatically falls back to the default value.

In [6]:
# Input Validation Function
# Ensures only valid values are passed to the prompt template
print("\n Input Validation:")

def validate_inputs(audience, tone):
    valid_audience = ["beginner", "intermediate", "expert"]
    valid_tone     = ["formal", "casual", "fun"]

    # If invalid audience → default to 'beginner'
    if audience not in valid_audience:
        audience = "beginner"

    # If invalid tone → default to 'formal'
    if tone not in valid_tone:
        tone = "formal"

    return audience, tone

# Test: 'kids' is not valid → becomes 'beginner'
#       'cool' is not valid → becomes 'formal'
print(validate_inputs("kids", "cool"))


 Input Validation:
('beginner', 'formal')


**Output Explanation:**

```
Input  : audience='kids', tone='cool'
Output : ('beginner', 'formal')
```
Both invalid values were automatically replaced with safe defaults — so the prompt always receives valid inputs.

---
## Example 5: Prompt Generator

### Combining Validation + Template = Smart Prompt Generator

Here we combine the **input validation function** with a **PromptTemplate** to create a complete, production-ready prompt generator.

**How it works:**
1. User provides `topic`, `audience`, and `tone`
2. `validate_inputs()` sanitizes the values
3. `teaching_prompt.format()` fills them into the template
4. Returns a clean, ready-to-use prompt

This is a great example of **modular design** — each function does one thing, and they work together cleanly.

In [7]:
# Prompt Generator Function
# Combines validation + template for a clean, modular prompt builder
print("\n Prompt Generator:")

teaching_prompt = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} teaching style"
)

def generate_prompt(topic, audience, tone):
    # Step 1: Validate inputs
    audience, tone = validate_inputs(audience, tone)
    # Step 2: Format and return the prompt
    return teaching_prompt.format(topic=topic, audience=audience, tone=tone)

# Test the generator
print(generate_prompt("Machine Learning", "beginner", "fun"))


 Prompt Generator:
Explain Machine Learning for beginner in a fun teaching style


---
## Example 6: Template Reusability

### Why Reusability Matters

One of the biggest advantages of `PromptTemplate` is **reusability**. Define the template once, and use it across hundreds of different inputs without repeating yourself.

This is especially useful when:
- Processing a batch of topics at once
- Building pipelines that apply the same prompt to multiple inputs
- Keeping prompt logic centralized and easy to update

In this example, we apply the same template to 5 different topics in a loop.

In [8]:
# Template Reusability Test
# Same template applied to multiple topics in a loop
print("\n Template Reusability Test:")

reuse_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple words"
)

# Apply the same template to 5 different topics
topics = ["AI", "Python", "ML", "DL", "Data Science"]

for t in topics:
    print(reuse_template.format(topic=t))


 Template Reusability Test:
Explain AI in simple words
Explain Python in simple words
Explain ML in simple words
Explain DL in simple words
Explain Data Science in simple words


---
## Summary

Here's a quick recap of everything we covered in this notebook:

| Example | Component | Key Takeaway |
|---|---|---|
| 1 | Prompt Template | Define reusable prompts with dynamic variables |
| 2 | Memory | Store and retrieve conversation history |
| 3 | Chat Prompt Template | Structure prompts with system + user roles |
| 4 | Input Validation | Sanitize user inputs before sending to LLM |
| 5 | Prompt Generator | Combine validation + template for modular design |
| 6 | Template Reusability | Apply one template to many inputs efficiently |

---

### Key Insight

LangChain's power comes from its **modularity** — each component does one thing well, and they all connect together cleanly. This makes it easy to build, test, and scale LLM-powered applications.

---
*End of Task 2 – Deep Technical Blog on LangChain*